In [1]:
!which python

/Users/ben/Code/codes/botorch-test/.venv/bin/python


In [4]:
!uv add botorch

Resolved 151 packages in 437ms                                       
Prepared 15 packages in 2.44s                                            
Installed 23 packages in 438ms                              
 + botorch==0.16.1
 + filelock==3.32.2
 + fsspec==2026.7.0
 + gpytorch==1.15.2
 + joblib==1.5.3
 + linear-operator==0.6.1
 + mpmath==1.3.0
 + multipledispatch==1.0.0
 + mypy-extensions==1.1.0
 + networkx==3.4.2
 + numpy==2.2.6
 + opt-einsum==3.4.0
 + pyre-extensions==0.0.32
 + pyro-api==0.1.2
 + pyro-ppl==1.9.1
 + scikit-learn==1.7.2
 + scipy==1.15.3
 + setuptools==83.0.0
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + torch==2.13.0
 + tqdm==4.70.0
 + typing-inspect==0.9.0


In [5]:
! uv tree

Resolved 151 packages in 6ms
botorch-test v0.1.0
├── botorch v0.16.1
│   ├── gpytorch v1.15.2
│   │   ├── linear-operator v0.6.1
│   │   │   ├── scipy v1.15.3
│   │   │   │   └── numpy v2.2.6
│   │   │   └── torch v2.13.0
│   │   │       ├── filelock v3.32.2
│   │   │       ├── fsspec v2026.7.0
│   │   │       ├── jinja2 v3.1.6
│   │   │       │   └── markupsafe v3.0.3
│   │   │       ├── networkx v3.4.2
│   │   │       ├── setuptools v83.0.0
│   │   │       ├── sympy v1.14.0
│   │   │       │   └── mpmath v1.3.0
│   │   │       └── typing-extensions v4.16.0
│   │   ├── mpmath v1.3.0
│   │   ├── scikit-learn v1.7.2
│   │   │   ├── joblib v1.5.3
│   │   │   ├── numpy v2.2.6
│   │   │   ├── scipy v1.15.3 (*)
│   │   │   └── threadpoolctl v3.6.0
│   │   └── scipy v1.15.3 (*)
│   ├── linear-operator v0.6.1 (*)
│   ├── multipledispatch v1.0.0
│   ├── pyre-extensions v0.0.32
│   │   ├── typing-extensions v4.16.0
│   │   └── typing-inspect v0.9.0
│   │       ├── mypy-extensions v1.1.0
│   │  

1. Fit a Gaussian Process model to data

In [6]:
import torch
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

# Double precision is highly recommended for GPs.
# See https://github.com/meta-pytorch/botorch/discussions/1444
train_X = torch.rand(10, 2, dtype=torch.double) * 2
Y = 1 - (train_X - 0.5).norm(dim=-1, keepdim=True)  # explicit output dimension
Y += 0.1 * torch.rand_like(Y)

gp = SingleTaskGP(
    train_X=train_X,
    train_Y=Y,
    input_transform=Normalize(d=2),
)
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
fit_gpytorch_mll(mll)

/Users/ben/Code/codes/botorch-test/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): LogNormalPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): LogNormalPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): RBFKernel(
      (lengthscale_prior): LogNormalPrior()
      (raw_lengthscale_constraint): GreaterThan(2.500E-02)
    )
    (outcome_transform): Standardize()
    (input_transform): Normalize()
  )
)

2. Construct an acquisition function

In [8]:
from botorch.acquisition import LogExpectedImprovement

logEI = LogExpectedImprovement(model=gp, best_f=Y.max())

3. Optimize the acquisition function

In [9]:
from botorch.optim import optimize_acqf

bounds = torch.stack([torch.zeros(2), torch.ones(2)]).to(torch.double)
candidate, acq_value = optimize_acqf(
    logEI, bounds=bounds, q=1, num_restarts=5, raw_samples=20,
)